In [ ]:
!pip install -q mlflow scikit-learn pandas numpy
!apt-get -qq install -y nodejs npm
!npm -g -s install localtunnel


In [9]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [ ]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "file:/content/mlruns"
print("MLFLOW_TRACKING_URI =", os.environ["MLFLOW_TRACKING_URI"])


In [ ]:
!npm install -g localtunnel
!mlflow ui --backend-store-uri file:/content/mlruns --host 0.0.0.0 --port 5000 &
!npx localtunnel --port 5000 --print-requests



changed 22 packages, and audited 23 packages in 3s

3 packages are looking for funding
  run `npm fund` for details

1 high severity vulnerability

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
INFO:     Uvicorn running on http://0.0.0.0:5000 (Press CTRL+C to quit)
INFO:     Started parent process [2512]
INFO:     Started server process [2515]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started server process [2517]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started ser

In [4]:

!npm install -g localtunnel

!mlflow ui --backend-store-uri file:/content/mlruns --host 0.0.0.0 --port 5000 &

!npx localtunnel --port 5000 --print-requests



changed 22 packages, and audited 23 packages in 2s

3 packages are looking for funding
  run `npm fund` for details

1 high severity vulnerability

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
INFO:     Uvicorn running on http://0.0.0.0:5000 (Press CTRL+C to quit)
INFO:     Started parent process [25228]
INFO:     Received SIGINT, exiting.
INFO:     Terminated child process [25230]
INFO:     Terminated child process [25231]
INFO:     Terminated child process [25232]
INFO:     Terminated child process [25233]
INFO:     Waiting for child process [25230]
INFO:     Star

In [8]:


X, y = datasets.load_diabetes(return_X_y=True)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


params = {"fit_intercept": True}


mlflow.set_experiment("diabetes_linear_regression")

with mlflow.start_run(run_name="linear_reg_skl_diabetes"):
    # Entrenamiento
    model = LinearRegression(**params)
    model.fit(X_train, y_train)

    # Predicción con metricas
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # Log params con metricas
    mlflow.log_params(params)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)


    X_train_df = pd.DataFrame(X_train)
    signature = infer_signature(X_train_df, model.predict(X_train_df))
    input_example = X_train_df.iloc[:5]

    #  modelo con el run
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )

print(f"MSE: {mse:.4f} | R2: {r2:.4f}")



2025/10/05 23:01:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MSE: 2900.1936 | R2: 0.4526


In [10]:

client = mlflow.tracking.MlflowClient()
exp_id = mlflow.get_experiment_by_name("diabetes_linear_regression").experiment_id
last_run = client.search_runs(
    experiment_ids=[exp_id],
    order_by=["attributes.start_time DESC"],
    max_results=1
)[0]
run_id = last_run.info.run_id

loaded_model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")


X_new = np.array([
    [ 0.038075906, 0.05068012,  0.06169621,  0.02187235, -0.0442235,  -0.03482076, -0.04340085, -0.00259226,  0.01990749, -0.01764613],
    [-0.001882016, 0.04445121, -0.05147406, -0.02632795, -0.00844872, -0.01916334,  0.07441156, -0.03949338, -0.06833155, -0.09220405]
])

preds = loaded_model.predict(X_new)
preds.tolist()


[210.74208602707498, 47.87926538468788]